# The α-Function Zoo (M7.2, planned v0.4.0)

> ⚠️ **DRAFT — M7.2 planned for `v0.4.0`.** The functionality this notebook will exercise has **not yet been implemented** in the engine. Today (v0.3.0) the calls in the "behavior today" section below raise `NotImplementedError` or panic with an `M7.x deferred` marker — that's intentional, not a bug. For what does work right now, see [`02_pure_component.ipynb`](02_pure_component.ipynb).

Ports the remaining **15 two-parameter cubic EOS variants** from `legacy/vb6/clsQbicsPure.cls:1719` — the polar Mathias-Naumann extensions, the PRSV (Stryjek-Vera) form with its component-specific K₁ parameter, the OL family that uses the family-table `h_k` coefficients, Berthelot, van-der-Waals-Adachi/Valderrama, and friends.


> 💾 **Hub sandbox notice — only applies if you're running this notebook on the hosted VLE JupyterLab.** If the VLE developers gave you a URL to a shared JupyterLab environment, that environment is an *educational sandbox*: edits you make to this notebook won't survive a container restart, the bundled `vle-thermo` version may lag PyPI, and any `pip install` you run inside this container is ephemeral (it vanishes when your session is culled). For real work, install `vle-thermo` in your own Jupyter environment with `pip install vle-thermo` and run the notebook there — see the [project README](https://github.com/miguelju/vle/blob/main/README.md). **If you opened this notebook in your own Jupyter, you can ignore this notice.**

## Setup (optional)

The cell below is **commented out by default**. Uncomment it if you want to use the latest `vle-thermo` released on PyPI instead of whatever version is currently installed in your kernel — this matters most for *this* notebook because the feature being demonstrated is **planned for a future release**, and you may already be on it by the time you read this.

In [1]:
# Optional: pull the latest vle-thermo from PyPI.
# Uncomment if you want the newest released version instead of
# whatever is currently in your kernel. On the hosted hub this
# install is ephemeral — it vanishes when your session is culled.
# %pip install --upgrade vle-thermo

## Why this is coming

M7.1 ships the **four α functions** that the Chapter IV validation cases actually use (PR, RKS, RK, VdW). That's enough to ship a real flash calculator at v0.5.0, but the VB6 legacy carries fifteen more α variants — polar Mathias-Naumann (`m`, `n`, `g` per-component), PRSV with its K₁ extra parameter, Graboski-Daubert, Lim's modifications, the OL family that pulls coefficients from the family `h_k` table. Each has its specific niche (PRSV for water, RKSGD for hydrocarbon-rich associating systems, …), and each follows the same Abbott form so the port is mostly translation + analytical-derivative bookkeeping.

## Planned scope (M7.2 → `v0.4.0`)

| Variant | VB6 line | α form sketch | Notes |
|---|---|---|---|
| Berth1899 | 1730 | 1/T_r | Trivial — temperature-modified VdW |
| VdWAda1984 | 1732 | 10^(m·(1−T_r)) | m a polynomial in ω |
| RKSGD1978 | 1738 | (1+m(1−√T_r))², new m(ω) | Graboski-Daubert |
| RKSL1997 | 1740 | RKS form, cubic-in-ω m(ω) | Lim modification |
| RP1978 | 1744 | PR form, cubic m(ω) | Redlich-Prausnitz |
| PRL1997 | 1746 | PR form, Lim m(ω) | |
| VdWVald1989 | 1748 | 1+(1−T_r)(m+n/T_r) | ω·Z_c-driven m, n |
| RKSmn1980 | 1753 | 1+(1−T_r)(m+n/T_r) | Polar, component-specific m, n |
| RKSATmn1995 | 1755 | exp(…) | Adachi-Tagawa-MN, three constants m/n/g |
| PRATmng1997 | 1757 | exp(…) | PR family of the above |
| PRMmn1989 | 1759 | exp(…) | PR Mathias-Massih-Naumann |
| PRSV1986 | 1762 | (1+(n+K₁(1+√T_r)(0.7−T_r))(1−√T_r))² | Stryjek-Vera; K₁ component-specific |
| VdWOL1998 | 1768 | T_r(1+Σ h_k·…) | OL family — pulls h_k from family-table |
| RKOL1998 | 1768 | T_r(1+Σ h_k·…) | Same form, RKS h_k |
| PROL1998 | 1768 | T_r(1+Σ h_k·…) | Same form, PR h_k |

Every variant requires an analytical dα/dT_r derivation (per CLAUDE.md *Algorithm Choices*). The component-specific polar parameters (`m_polar`, `n_polar`, `g_polar`, `prsv_k1`) are already on the `Component` struct — M7.2 just plugs in the formula and the corresponding test against a central-difference oracle.

## Behavior today (v0.3.0)

The cell below calls the planned feature **through the existing engine binding**. Today it raises the deferred-stub error so you can see exactly what M7.2 will eventually replace. Once that milestone ships in `v0.4.0`, the same cell will produce real numbers — and this banner will go away.

In [2]:
from vle._engine import CubicEos, eos_alpha

# RKSGD1978 is one of the M7.2-deferred variants.
try:
    eos_alpha(CubicEos.RKSGD1978, 0.85, 0.252)
    print('ERROR: expected a panic but got a value')
except BaseException as exc:
    print(f'{type(exc).__name__}: {exc}')

PanicException: not implemented: M7.2 deferred: alpha(RKSGD1978) not yet ported — see legacy/vb6/clsQbicsPure.cls:1719



thread '<unnamed>' (1701843) panicked at engine/src/eos.rs:321:13:
not implemented: M7.2 deferred: alpha(RKSGD1978) not yet ported — see legacy/vb6/clsQbicsPure.cls:1719
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace


## References

- **ROADMAP.md** — the live status of this sub-milestone. [`ROADMAP.md`](https://github.com/miguelju/vle/blob/main/ROADMAP.md)
- **MODERNIZATION_PLAN.md** — phase-level technical scope. [`MODERNIZATION_PLAN.md`](https://github.com/miguelju/vle/blob/main/MODERNIZATION_PLAN.md)
- **v0.3.0 functional notebook** — what works today. [`02_pure_component.ipynb`](02_pure_component.ipynb)